<a href="https://colab.research.google.com/github/Kadidjah/LFA/blob/main/SistemaSE_Orientador_UDF_Solucao_Comentada.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Orientador UDF — Solução comentada
**Inteligência Artificial · Profa. Kadidja Valéria** (03/11)

No Colab, use **Arquivo → Fazer upload de notebook** e execute as células em ordem. Sem bibliotecas adicionais. As optativas e os critérios são fictícios.

## 1. Base de conhecimento
As perguntas representam três fatos: interesse por matemática, programação e análise de dados. Cada regra associa condições a uma optativa. R1 exige interesse por matemática e programação.

In [1]:
PERGUNTAS = {
    "matematica": "Você gosta de matemática?",
    "programacao": "Você gosta de programar?",
    "dados": "Você gosta de analisar dados?",
}

# Cada condição guarda o nome do fato e o valor esperado.
REGRAS = [
    ("R1", "Introdução à IA", {"matematica": True, "programacao": True}),
    ("R2", "Análise de Dados", {"dados": True, "matematica": True}),
    ("R3", "Desenvolvimento de Software", {"programacao": True, "dados": False}),
    ("R4", "Visualização de Dados", {"dados": True, "matematica": False}),
    ("R5", "Fundamentos de Lógica", {"matematica": True, "programacao": False}),
]

## 2. Motor de inferência
O motor parte da optativa escolhida e verifica as premissas da regra. O dicionário de fatos guarda as respostas. Uma premissa incompatível rejeita a regra. Uma resposta desconhecida, sem incompatibilidades, deixa a consulta inconclusiva. O encadeamento para trás tem um nível: as premissas são fatos, sem subobjetivos.

In [2]:
def consultar(objetivo, fatos, perguntar):
    """Parte do objetivo e verifica as premissas da regra correspondente."""
    codigo, _, condicoes = next(r for r in REGRAS if r[1] == objetivo)
    rastro = []
    pendentes = False
    for fato, esperado in condicoes.items():
        # Pergunte somente se o fato ainda não estiver na memória.
        if fato not in fatos:
            fatos[fato] = perguntar(PERGUNTAS[fato])
        valor = fatos[fato]
        rotulo = {True: "sim", False: "não", None: "não informado"}
        rastro.append(f"{PERGUNTAS[fato]} {rotulo[valor]} (esperado: {rotulo[esperado]})")
        # Informação ausente não equivale a uma resposta negativa.
        if valor is None:
            pendentes = True
        elif valor != esperado:
            return codigo, "Critério não atendido neste exemplo.", rastro
    if pendentes:
        return codigo, "Informação insuficiente para recomendar.", rastro
    return codigo, f"Recomendação didática: {objetivo}.", rastro

## 3. Simulação guiada
Para Introdução à IA, simulamos duas respostas afirmativas. Observe que R1 não exige perguntar sobre análise de dados.

In [3]:
respostas = iter([True, True])
regra, resultado, rastro = consultar("Introdução à IA", {}, lambda _: next(respostas))
print(regra, resultado)
print(*rastro, sep="\n")

R1 Recomendação didática: Introdução à IA.
Você gosta de matemática? sim (esperado: sim)
Você gosta de programar? sim (esperado: sim)


## 4. Testes automáticos
Confirme as cinco regras. Em seguida, compare a rejeição de R1 com uma consulta inconclusiva. Fatos conhecidos não devem gerar novas perguntas.

In [4]:
def nao_perguntar(pergunta):
    raise AssertionError(f"Pergunta inesperada: {pergunta}")

for codigo, objetivo, fatos in REGRAS:
    regra, resultado, _ = consultar(objetivo, dict(fatos), nao_perguntar)
    assert regra == codigo
    assert resultado == f"Recomendação didática: {objetivo}."
    print(f"{codigo}: confirmação verificada")

for fatos, esperado in [
    ({"matematica": False}, "Critério não atendido neste exemplo."),
    ({"matematica": None, "programacao": True}, "Informação insuficiente para recomendar."),
]:
    _, resultado, _ = consultar("Introdução à IA", fatos, nao_perguntar)
    assert resultado == esperado
    print(resultado)

R1: confirmação verificada
R2: confirmação verificada
R3: confirmação verificada
R4: confirmação verificada
R5: confirmação verificada
Critério não atendido neste exemplo.
Informação insuficiente para recomendar.


## 5. Consulta interativa
A interface valida as respostas e mostra o rastro. Execute a última célula de código e escolha 1. Compare três execuções: **s, s**, **n** e **?, s**.

In [6]:
def perguntar_console(pergunta):
    while True:
        resposta = input(f"{pergunta} [s/n/?] ").strip().lower()
        if resposta in ("s", "n", "?"):
            return {"s": True, "n": False, "?": None}[resposta]
        print("Digite s, n ou ? (não informado).")


def main():
    print("Orientador UDF | Exemplo didático com optativas fictícias")
    print("Escolha a optativa que deseja investigar:")
    for i, (_, nome, _) in enumerate(REGRAS, 1):
        print(f"{i}. {nome}")
    opcoes = {str(i): nome for i, (_, nome, _) in enumerate(REGRAS, 1)}
    while True:
        escolha = input("Opção: ").strip()
        if escolha in opcoes:
            break
        print("Escolha um número de 1 a 5.")
    codigo, resultado, rastro = consultar(opcoes[escolha], {}, perguntar_console)
    print(f"\nRegra investigada: {codigo}")
    for passo in rastro:
        print(f"- {passo}")
    print(resultado)

In [ ]:
main()

## 6. Desafio do estudante
1. Defina um domínio, um público e uma decisão.
2. Escreva três fatos observáveis e cinco regras SE–ENTÃO, uma por objetivo. Indique a origem dos critérios.
3. Simule no papel a ordem das perguntas para um objetivo e adapte a base de conhecimento.
4. Teste um caso confirmado, um rejeitado e um inconclusivo. Entregue o notebook com regras, entradas, resultados e uma limitação.

Material-base: *Sistemas Especialistas*, atividade Hands On: Modelando o seu Sistema.